# Conventional education-access features: hierarchical clustering

Same structure as `feats_clustering.ipynb`, with education centres (CSIMBOL `020903`, `020905`).

1. **Cell 0** computes 35 features (the 25 used for health plus 10 education-specific ones) for the 17 cities and writes `city_feature_matrix_education.csv` (about 1–2 min).
2. **Cell 1** runs hierarchical clustering at k = 2 on `FEATURE_SET` (`"all"`, `"base"` or `"education"`) and draws the dendrogram and silhouette, then compares with the education superlevel-H0 TDA split.
3. **Cell 2** analyses the number of clusters (validity indices and bootstrap stability).
4. **Cell 3** clusters on the features that differ between the TDA groups (BH q < 0.05) plus a few non-significant ones, and tests the agreement against a label-permutation null (selection is redone in every permutation). Outputs go to `feature_clustering_education/selected/`.

Outputs go to `feature_clustering_education/<FEATURE_SET>/`, so the health results in `feature_clustering/` are not overwritten.

In [ ]:
"""
City x feature matrix of conventional EDUCATION-access metrics (25 base + 10 education-specific = 35 features, 17 cities).

Routing logic follows level_set_both.ipynb, with one difference: walking time
is routed to MGN *block centroids* instead of momepy tessellation cells, so that
each value links 1:1 to the DANE 2018 block table.
  - Study area: DANE urban perimeters that intersect the city's walking network.
  - Education centres: CSIMBOL codes 020903, 020905, inside the study area + 1 km buffer.
  - Edge speed: slope-velocity table (5 km/h on flat edges); connectors at 5 km/h.
  - Walking time: multi-source Dijkstra; block time = network time + snap distance / 5 km/h.
  - Population weights: DANE 2018 block population; times capped at 60 min.

Usage (terminal):   python city_feature_matrix.py
Usage (notebook):   from city_feature_matrix import build_feature_matrix
                    F = build_feature_matrix()          # DataFrame, rows = cities
"""
from pathlib import Path
import time

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.sparse import coo_matrix, csr_matrix, vstack
from scipy.sparse.csgraph import dijkstra
from scipy.spatial import cKDTree

# ----------------------------------------------------------------------------- paths / parameters
cwd = Path.cwd().resolve()
REPO = next(
    path for path in (cwd, cwd / "pedestrian_accessibility", *cwd.parents)
    if (path / "data/input/shape").is_dir()
)
SHAPE = REPO / "data/input/shape"
TABLE = REPO / "data/input/table"
RESULTS = REPO / "level_sets_blocks/results/experiment_1_complete_cities"
RESULTS.mkdir(parents=True, exist_ok=True)
OUT_CSV = RESULTS / "city_feature_matrix_education.csv"

CITIES = ["barranquilla", "bogota", "bucaramanga", "cali", "cartagena", "cucuta", "ibague",
          "manizales", "medellin", "monteria", "neiva", "pasto", "pereira", "popayan",
          "santa_marta", "valledupar", "villavicencio"]

CENTRE_CODES = ["020903", "020905"]          # education centres (as in level_set_both.ipynb)


BUFFER_M = 1000                 # external buffer for centres (m)
CONNECTOR_SPEED = 5000 / 60     # m/min (5 km/h) for block->network and centre->network connectors
T_CAP = 60.0                    # minutes; same cap as the persistence filtration
BATCH = 40                      # centres per Dijkstra batch (memory control)
SFCA_CATCH = 15.0               # minutes; catchment for the two-step floating catchment area (2SFCA)

FEATURES = {
    # supply
    "centres_per100k":       "Education centres per 100k residents",
    "centres_per_km2":       "Education centres per km² of study area",
    # walking-time distribution (population-weighted)
    "t_p25":                 "25th percentile walking time (min)",
    "t_p50":                 "Median walking time (min)",
    "t_p90":                 "90th percentile walking time (min)",
    "share_lt10":            "Share of residents < 10 min from a centre",
    "share_lt5":             "Share of residents < 5 min from a centre",
    "t_iqr":                 "Interquartile range of walking time, p75 - p25 (min)",
    "share_gt30":            "Share of residents > 30 min from a centre (coverage gap)",
    # redundancy / choice
    "share_ge3_hc15":        "Share of residents with >= 3 centres within 15 min",
    "n_hc15_pw":             "Mean number of centres reachable within 15 min (per resident)",
    "n_hc30_pw":             "Mean number of centres reachable within 30 min (per resident)",
    "share_no_hc15":         "Share of residents with no centre within 15 min",
    "share_ge2_hc15":        "Share of residents with >= 2 centres within 15 min (backup coverage)",
    "backup_penalty_p50":    "Median extra walking time if the nearest centre closed (2nd - 1st, min)",
    "t2_p50":                "Median walking time to the 2nd-nearest centre (min)",
    # catchments
    "catchment_load_pw_p50": "Median population sharing a resident's nearest centre",
    "catchment_pop_cv":      "Coefficient of variation of catchment population across centres",
    "sfca15_p50":            "2SFCA accessibility, 15-min catchment: median centres per 10k residents",
    "area_cov15":            "Share of inhabited block area within 15 min of a centre (spatial coverage)",
    # geometry / network
    "euclid_p50_m":          "Median straight-line distance to the nearest centre (m)",
    "share_beyond_800m_euclid": "Share of residents > 800 m straight-line from a centre",
    "share_net_grade_gt10":  "Share of network length with grade > 10%",
    # inequality / equity
    "gini_time":             "Gini coefficient of walking time (population-weighted)",
    "gap_bajo_alto":         "Mean walking time, low strata (1-2) minus high strata (5-6) (min)",
    # ---- education-specific additions (10) ----
    # terrain cost of walking (children walk slowly uphill)
    "slope_penalty_p50":     "Median extra walking time caused by slope (sloped minus flat 5 km/h, min)",
    "share_slope_pen_gt2":   "Share of residents losing > 2 min to slope on the way to the nearest centre",
    "circuity_p50":          "Median network / straight-line distance to nearest centre (blocks > 200 m away)",
    "intersection_density":  "Street intersections (degree >= 3) per km² of study area",
    # spatial arrangement of the centres themselves
    "clark_evans_R":         "Clark-Evans nearest-neighbour ratio of centres (< 1 clustered, 1 random, > 1 dispersed)",
    "centre_isolation":      "Share of centres with no other centre within 15 min walk",
    # composition by code (020903 vs 020905)
    "share_code_020905":     "Share of centres with CSIMBOL 020905",
    "t_p50_code_020905":     "Median walking time to the nearest CSIMBOL-020905 centre (min)",
    # equity of supply and link to outcomes
    "supply_equity_low":     "Share of centres in low-strata blocks / share of residents in low strata",
    "rho_time_inasist":      "Population-weighted rank correlation, block walking time vs non-attendance age 5-14",
}


# ----------------------------------------------------------------------------- helpers
def wquantile(x, w, q):
    """Weighted quantile (linear interpolation on the cumulative weight)."""
    o = np.argsort(x)
    x, w = x[o], w[o]
    c = np.cumsum(w) / w.sum()
    return float(np.interp(q, c, x))


def wgini(x, w):
    """Population-weighted Gini coefficient."""
    o = np.argsort(x)
    x, w = x[o], w[o]
    p = np.concatenate([[0], np.cumsum(w) / w.sum()])
    L = np.concatenate([[0], np.cumsum(w * x) / (w * x).sum()])
    return float(1 - np.sum((p[1:] - p[:-1]) * (L[1:] + L[:-1])))


def wspearman(x, y, w):
    """Population-weighted Pearson correlation of ranks."""
    rx, ry = pd.Series(x).rank().to_numpy(), pd.Series(y).rank().to_numpy()
    mx, my = np.average(rx, weights=w), np.average(ry, weights=w)
    cov = np.average((rx - mx) * (ry - my), weights=w)
    return float(cov / np.sqrt(np.average((rx - mx) ** 2, weights=w) * np.average((ry - my) ** 2, weights=w)))


def edge_speeds(edges, speed_table):
    """Slope-dependent walking speed (km/h) per edge, as in the notebook."""
    sp = np.zeros(len(edges))
    g = edges["grade_abs"].to_numpy()
    for iv in speed_table.itertuples(index=False):
        sp[(g > iv.s1) & (g <= iv.s2)] = iv.speed
    sp[g == 0] = 5.0
    sp[g > speed_table["s2"].max()] = speed_table.iloc[0]["speed"]
    return sp


# ----------------------------------------------------------------------------- shared inputs (read once)
def load_shared():
    speed_table = pd.read_csv(TABLE / "slope_velocity_data.csv", sep=";")
    speed_table.columns = [c.strip("\ufeff") for c in speed_table.columns]
    perims = gpd.read_file(SHAPE / "Colombia_urban_perimeters_name_pop/Colombia_urban_perimeters_name_pop.shp")
    dane = pd.read_csv(TABLE / "DANE_2018_personas_manz.txt",
                       usecols=["MANZ_CCNCT", "poblacion", "estrato_bajo", "estrato_alto", "inasistencia_5a14"],
                       dtype={"MANZ_CCNCT": "string"}, low_memory=False)
    return speed_table, perims, dane


# ----------------------------------------------------------------------------- per-city computation
def city_features(city, speed_table, perims_all, dane):
    t0 = time.time()

    # --- network
    edge_path = next((SHAPE / "edges").glob(f"{city}-*_edges.shp"))
    nid = edge_path.stem.removesuffix("_edges").rsplit("-", 1)[1]
    raw_nodes = gpd.read_file(SHAPE / "nodes" / f"{city}-{nid}_nodes.shp")
    crs = raw_nodes.to_crs("EPSG:4326").estimate_utm_crs()
    edges = gpd.read_file(edge_path).to_crs(crs)
    nodes = raw_nodes.to_crs(crs)
    edges["from"] = edges["from"].astype(str)
    edges["to"] = edges["to"].astype(str)
    nodes["osmid"] = nodes["osmid"].astype(str)
    edges["w"] = edges["length"] / (edge_speeds(edges, speed_table) * 1000 / 60)   # minutes

    # --- study area = urban perimeters touched by the network
    perims = perims_all.to_crs(crs)
    hits = gpd.sjoin(edges[["geometry"]], perims[["ID_mpio", "geometry"]], predicate="intersects")
    city_geom = perims[perims["ID_mpio"].isin(hits["ID_mpio"].unique())].geometry.union_all()
    area_km2 = city_geom.area / 1e6

    # --- blocks + DANE attributes
    poly = gpd.GeoDataFrame({"geometry": [city_geom]}, crs=crs)
    cand = gpd.read_file(SHAPE / "MANZANA/MGN_URB_MANZANA.shp",
                         mask=poly.to_crs("EPSG:4326").geometry.iloc[0]).to_crs(crs)
    blocks = (gpd.sjoin(cand, poly, predicate="intersects").drop(columns="index_right")
              .drop_duplicates("MANZ_CCNCT").reset_index(drop=True))
    blocks["MANZ_CCNCT"] = blocks["MANZ_CCNCT"].astype("string")
    blocks = blocks.merge(dane, on="MANZ_CCNCT", how="left", validate="one_to_one")

    # --- health centres in study area + buffer
    buf = city_geom.buffer(BUFFER_M)
    eq = gpd.read_file(SHAPE / "colombia_equipments_ID/colombia_equipments_ID.shp",
                       mask=gpd.GeoSeries([buf], crs=crs).to_crs("EPSG:4326").iloc[0]).to_crs(crs)
    eq["CSIMBOL"] = eq["CSIMBOL"].astype("string").str.strip()
    hc = eq[eq["CSIMBOL"].isin(CENTRE_CODES) & eq.geometry.intersects(buf)].reset_index(drop=True)
    H = len(hc)
    hxy = np.column_stack([hc.geometry.x, hc.geometry.y])

    # --- graph: undirected, parallel edges -> minimum time, no self-loops
    ids = pd.Index(pd.unique(np.concatenate([edges["from"].to_numpy(), edges["to"].to_numpy()])))
    u, v = ids.get_indexer(edges["from"]), ids.get_indexer(edges["to"])
    e = pd.DataFrame({"a": np.minimum(u, v), "b": np.maximum(u, v), "w": edges["w"].to_numpy()})
    e = e[e.a != e.b].groupby(["a", "b"], as_index=False)["w"].min()
    N = len(ids)

    nn = nodes.drop_duplicates("osmid").set_index("osmid").reindex(ids)
    have = nn.geometry.notna().to_numpy()
    nxy = np.column_stack([nn.geometry.x.to_numpy(), nn.geometry.y.to_numpy()])
    tree_idx = np.flatnonzero(have)
    tree = cKDTree(nxy[have])

    # one virtual source node per centre (keeps the connector cost and the centre identity)
    hd, hp = tree.query(hxy)
    hnode = tree_idx[hp]
    hcon = np.maximum(hd / CONNECTOR_SPEED, 1e-9)          # csgraph treats 0 as "no edge"
    rows = np.concatenate([e.a, e.b, N + np.arange(H)])
    cols = np.concatenate([e.b, e.a, hnode])
    vals = np.concatenate([e.w, e.w, hcon])
    G = coo_matrix((vals, (rows, cols)), shape=(N + H, N + H)).tocsr()

    # nearest centre: time + which centre
    dist, _, src = dijkstra(G, directed=True, indices=N + np.arange(H),
                            min_only=True, return_predecessors=True)
    # same graph with every edge at flat 5 km/h (isolates the slope cost; x speed = network metres)
    ef = pd.DataFrame({"a": np.minimum(u, v), "b": np.maximum(u, v), "L": edges["length"].to_numpy(float)})
    ef = ef[ef.a != ef.b].groupby(["a", "b"], as_index=False)["L"].min()
    Gf = coo_matrix((np.concatenate([ef.L, ef.L, hd]) / CONNECTOR_SPEED,
                     (np.concatenate([ef.a, ef.b, N + np.arange(H)]), np.concatenate([ef.b, ef.a, hnode]))),
                    shape=(N + H, N + H)).tocsr()
    Gf.data = np.maximum(Gf.data, 1e-9)
    dist_flat = dijkstra(Gf, directed=True, indices=N + np.arange(H), min_only=True)
    # nearest centre of code 020905
    code = hc["CSIMBOL"].to_numpy()
    sel = np.flatnonzero(code == "020905")
    dist_05 = (dijkstra(G, directed=True, indices=N + sel, min_only=True) if len(sel)
               else np.full(N + H, np.inf))
    # --- attach blocks to the network (centroid -> nearest network node)
    cen = blocks.geometry.centroid
    bxy = np.column_stack([cen.x, cen.y])
    bd, bp = tree.query(bxy)
    bn = tree_idx[bp]
    snap_t = bd / CONNECTOR_SPEED
    pop_all = blocks["poblacion"].fillna(0).to_numpy(float)

    # counts within 15/30 min, 2nd-nearest time and the centre->block 15-min reach matrix
    # (batched, truncated at T_CAP)
    reach_rows = []
    c2c15 = np.zeros(H, np.int32)                          # other centres within 15 min of each centre
    n15 = np.zeros(N, np.int32)
    n30 = np.zeros(N, np.int32)
    best = np.full(N, np.inf)
    second = np.full(N, np.inf)
    for i0 in range(0, H, BATCH):
        D = dijkstra(G, directed=True, indices=N + np.arange(i0, min(H, i0 + BATCH)), limit=T_CAP)[:, :N]
        n15 += (D <= 15).sum(0)
        n30 += (D <= 30).sum(0)
        stack = np.vstack([best, second, D])
        stack.sort(axis=0)
        best, second = stack[0], stack[1]
        reach_rows.append(csr_matrix((D[:, bn] + snap_t[None, :]) <= SFCA_CATCH))   # centres x blocks
        Dc = D[:, hnode] + hcon[None, :]
        Dc[np.arange(Dc.shape[0]), np.arange(i0, i0 + Dc.shape[0])] = np.inf   # exclude self
        c2c15[i0:i0 + Dc.shape[0]] = (Dc <= 15).sum(1)

    # --- 2SFCA (Luo & Wang 2003), binary 15-min catchment, every centre has capacity 1
    reach = vstack(reach_rows).tocsr().astype(float)       # H x n_blocks
    demand = reach @ pop_all                                # residents within 15 min of each centre
    ratio = np.divide(1.0, demand, out=np.zeros(H), where=demand > 0)
    sfca = (reach.T @ ratio) * 1e4                          # centres per 10k residents, per block

    t_raw = dist[bn] + snap_t
    reached = np.isfinite(t_raw)

    t = np.minimum(t_raw, T_CAP)[reached]
    t2 = np.minimum(second[bn] + snap_t, T_CAP)[reached]
    hc_id = (src[bn] - N)[reached]
    k15 = n15[bn][reached]
    k30 = n30[bn][reached]
    d_euc = cKDTree(hxy).query(bxy)[0][reached]
    pop = pop_all[reached]
    sfca_r = sfca[reached]
    barea = blocks.geometry.area.to_numpy()[reached]
    low = blocks["estrato_bajo"].fillna(0).to_numpy(float)[reached]
    t_flat = (dist_flat[bn] + snap_t)[reached]
    t_05 = np.minimum(dist_05[bn] + snap_t, T_CAP)[reached]
    d_net = t_flat * CONNECTOR_SPEED                         # network metres to the nearest centre
    inas = blocks["inasistencia_5a14"].to_numpy(float)[reached]
    high = blocks["estrato_alto"].fillna(0).to_numpy(float)[reached]

    m = pop > 0                                   # inhabited, reachable blocks
    tp, pp = t[m], pop[m]
    total_pop = pop.sum()                         # population of reachable blocks

    # catchments: population whose nearest centre is each centre (zeros kept)
    load = np.bincount(hc_id[m], weights=pp, minlength=H)

    # equity: strata-weighted mean times (block strata columns are population shares)
    wl, wh = pp * low[m], pp * high[m]
    t_low = np.average(tp, weights=wl) if wl.sum() > 0 else np.nan
    t_high = np.average(tp, weights=wh) if wh.sum() > 0 else np.nan

    # terrain (grade_abs is a fraction: 0.10 = 10 %)
    L = edges["length"].to_numpy(float)
    gr = edges["grade_abs"].to_numpy(float)

    # --- education-specific additions
    slope_pen = np.minimum(t[m], T_CAP) - np.minimum(t_flat[m], T_CAP)
    far = d_euc[m] > 200
    circ = d_net[m][far] / d_euc[m][far]
    # intersections inside the study area
    deg = np.bincount(np.concatenate([e.a, e.b]), minlength=N)
    inside_nodes = gpd.GeoSeries(gpd.points_from_xy(nxy[have, 0], nxy[have, 1]), crs=crs).within(city_geom).to_numpy()
    n_int = int(((deg[have] >= 3) & inside_nodes).sum())
    # centre pattern (centres inside the study area only)
    cin = hc.geometry.within(city_geom).to_numpy()
    nin = int(cin.sum())
    if nin >= 2:
        nn_d = cKDTree(hxy[cin]).query(hxy[cin], k=2)[0][:, 1]
        ce_R = nn_d.mean() / (0.5 * np.sqrt(city_geom.area / nin))
    else:
        ce_R = np.nan
    # strata of the block nearest to each centre (inside the study area)
    low_all = blocks["estrato_bajo"].fillna(0).to_numpy(float)
    c_low = low_all[cKDTree(bxy).query(hxy[cin])[1]] if nin else np.array([np.nan])
    pop_low = (pp * low[m]).sum() / pp.sum()
    okr = np.isfinite(inas[m])

    f = {
        "centres_per100k":       H / total_pop * 1e5,
        "centres_per_km2":       H / area_km2,
        "t_p25":                 wquantile(tp, pp, 0.25),
        "t_p50":                 wquantile(tp, pp, 0.50),
        "t_p90":                 wquantile(tp, pp, 0.90),
        "share_lt10":            pp[tp < 10].sum() / pp.sum(),
        "share_lt5":             pp[tp < 5].sum() / pp.sum(),
        "t_iqr":                 wquantile(tp, pp, 0.75) - wquantile(tp, pp, 0.25),
        "share_gt30":            pp[tp > 30].sum() / pp.sum(),
        "share_ge3_hc15":        pp[k15[m] >= 3].sum() / pp.sum(),
        "n_hc15_pw":             np.average(k15[m], weights=pp),
        "n_hc30_pw":             np.average(k30[m], weights=pp),
        "share_no_hc15":         pp[k15[m] == 0].sum() / pp.sum(),
        "share_ge2_hc15":        pp[k15[m] >= 2].sum() / pp.sum(),
        "backup_penalty_p50":    wquantile(t2[m] - tp, pp, 0.50),
        "t2_p50":                wquantile(t2[m], pp, 0.50),
        "catchment_load_pw_p50": wquantile(load[hc_id[m]], pp, 0.50),
        "catchment_pop_cv":      load.std() / load.mean(),
        "sfca15_p50":            wquantile(sfca_r[m], pp, 0.50),
        "area_cov15":            barea[m][tp <= 15].sum() / barea[m].sum(),
        "euclid_p50_m":          wquantile(d_euc[m], pp, 0.50),
        "share_beyond_800m_euclid": pp[d_euc[m] > 800].sum() / pp.sum(),
        "share_net_grade_gt10":  L[gr > 0.10].sum() / L.sum(),
        "gini_time":             wgini(tp, pp),
        "gap_bajo_alto":         t_low - t_high,
        # ---- education-specific additions
        "slope_penalty_p50":     wquantile(slope_pen, pp, 0.50),
        "share_slope_pen_gt2":   pp[slope_pen > 2].sum() / pp.sum(),
        "circuity_p50":          wquantile(circ, pp[far], 0.50) if far.any() else np.nan,
        "intersection_density":  n_int / area_km2,
        "clark_evans_R":         ce_R,
        "centre_isolation":      float((c2c15 == 0).mean()),
        "share_code_020905":     len(sel) / H,
        "t_p50_code_020905":     wquantile(t_05[m], pp, 0.50),
        "supply_equity_low":     np.nanmean(c_low) / pop_low if pop_low > 0 else np.nan,
        "rho_time_inasist":      wspearman(tp[okr], inas[m][okr], pp[okr]),
    }
    print(f"{city:<14} blocks={len(blocks):>6}  centres={H:>4}  unreached={int((~reached).sum()):>4}  "
          f"{time.time() - t0:5.1f}s", flush=True)
    return f


# ----------------------------------------------------------------------------- driver
def build_feature_matrix(cities=CITIES, out_csv=OUT_CSV):
    speed_table, perims_all, dane = load_shared()
    F = pd.DataFrame({c: city_features(c, speed_table, perims_all, dane) for c in cities}).T
    F.index.name = "city"
    F = F[list(FEATURES)]
    if out_csv is not None:
        F.to_csv(out_csv)
        pd.Series(FEATURES, name="description").rename_axis("feature").to_csv(
            Path(out_csv).with_name("city_feature_dictionary_education.csv"))
    return F


if True:   # run when the cell is executed
    F = build_feature_matrix()
    pd.set_option("display.width", 200)
    print(F.round(3))
    # Standardised version (z-scores per feature), e.g. for clustering / PCA:
    Z = (F - F.mean()) / F.std(ddof=0)
    Z.to_csv(OUT_CSV.with_name("city_feature_matrix_education_zscores.csv"))

In [ ]:
# ============================================================================
# CELL 1 — Hierarchical clustering of cities on conventional EDUCATION features (k = 2)
# ============================================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram, cophenet
from scipy.spatial.distance import pdist
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score

# ---- parameters -------------------------------------------------------------
FEATURE_CSV = OUT_CSV
K = 2
METHOD = "average"        # "ward" (Euclidean only), "average", "complete", "single"
METRIC = "euclidean"   # used by pdist when METHOD != "ward"
FEATURE_SET = "all"    # "all" = 35 features, "base" = the 25 health-derived ones, "education" = the 10 new ones
USE_PCA = None         # None = z-scored features; e.g. 0.90 = keep PCs explaining 90 % variance
COLORS = ["#1b7837", "#e08214"]          # cluster 1, cluster 2
ABOVE = "#9e9e9e"                        # links joining different clusters
OUT_DIR = FEATURE_CSV.parent / "feature_clustering_education" / FEATURE_SET
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- data ---------------------------------------------------------------------
F = pd.read_csv(FEATURE_CSV, index_col="city")
EDU10 = ["slope_penalty_p50", "share_slope_pen_gt2", "circuity_p50", "intersection_density", "clark_evans_R",
          "centre_isolation", "share_code_020905", "t_p50_code_020905", "supply_equity_low", "rho_time_inasist"]
if FEATURE_SET == "base":
    F = F.drop(columns=EDU10)
elif FEATURE_SET == "education":
    F = F[EDU10]
print(f"feature set = {FEATURE_SET} ({F.shape[1]} features)")
assert F.notna().all().all(), F.columns[F.isna().any()].tolist()
cities = F.index.tolist()
Xz = (F - F.mean()) / F.std(ddof=0)                     # z-score: every feature weighs the same

if USE_PCA:
    from sklearn.decomposition import PCA
    pca = PCA(n_components=USE_PCA, svd_solver="full").fit(Xz)
    X = pca.transform(Xz)
    print(f"PCA: {pca.n_components_} components, {pca.explained_variance_ratio_.sum():.1%} variance")
else:
    X = Xz.to_numpy()

# ---- clustering ----------------------------------------------------------------
if METHOD == "ward":
    Z = linkage(X, method="ward")                       # Ward on Euclidean distances
    D = pdist(X, "euclidean")
else:
    D = pdist(X, METRIC)
    Z = linkage(D, method=METHOD)

labels = fcluster(Z, K, criterion="maxclust")
coph_r = cophenet(Z, D)[0]
sil = silhouette_score(X, labels, metric="euclidean" if METHOD == "ward" else METRIC)
sil_i = silhouette_samples(X, labels, metric="euclidean" if METHOD == "ward" else METRIC)
memb = pd.DataFrame({"cluster": labels, "silhouette": sil_i}, index=cities)

print(f"method = {METHOD}, k = {K}")
print(f"mean silhouette     = {sil:.3f}")
print(f"cophenetic corr.    = {coph_r:.3f}")
print("cluster sizes       =", memb.cluster.value_counts().sort_index().to_dict())

# ---- dendrogram colouring: a link gets a cluster colour only if all its leaves share that cluster
n = len(cities)
leaf_sets = {i: {labels[i]} for i in range(n)}
for j, (a, b, _, _) in enumerate(Z):
    leaf_sets[n + j] = leaf_sets[int(a)] | leaf_sets[int(b)]
link_color = lambda k: COLORS[next(iter(leaf_sets[k])) - 1] if len(leaf_sets[k]) == 1 else ABOVE
cut = (Z[-K, 2] + Z[-K + 1, 2]) / 2                     # height between the (K-1)th and Kth last merges

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8), gridspec_kw={"width_ratios": [1.6, 1]})
R = dendrogram(Z, labels=[c.replace("_", " ").title() for c in cities], ax=ax1,
               link_color_func=link_color, leaf_rotation=90, leaf_font_size=9)
for tick, idx in zip(ax1.get_xticklabels(), R["leaves"]):
    tick.set_color(COLORS[labels[idx] - 1])
ax1.axhline(cut, ls="--", lw=0.8, color="k")
ax1.set_ylabel("Ward linkage height" if METHOD == "ward" else f"{METHOD} linkage height")
ax1.set_title(f"Cities clustered on {F.shape[1]} education features ({FEATURE_SET})\n"
              f"{METHOD} linkage, k = {K}; silhouette = {sil:.2f}, cophenetic r = {coph_r:.2f}", loc="left", fontsize=10)
for s in ("top", "right"):
    ax1.spines[s].set_visible(False)

# silhouette plot (one bar per city, grouped by cluster)
order = memb.sort_values(["cluster", "silhouette"], ascending=[True, False])
y = np.arange(len(order))
ax2.barh(y, order.silhouette, color=[COLORS[c - 1] for c in order.cluster])
ax2.set_yticks(y, [c.replace("_", " ").title() for c in order.index], fontsize=8)
ax2.invert_yaxis()
ax2.axvline(sil, ls="--", lw=0.8, color="k")
ax2.axvline(0, lw=0.6, color="k")
ax2.set_xlabel("Silhouette width")
ax2.set_title(f"Per-city silhouette (mean = {sil:.2f})", loc="left", fontsize=10)
for s in ("top", "right"):
    ax2.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / f"dendrogram_features_{METHOD}_k{K}.png", dpi=300, bbox_inches="tight")
plt.show()

# ---- what separates the two clusters? -----------------------------------------
prof = pd.DataFrame({f"median_c{c}": F[labels == c].median() for c in range(1, K + 1)})
prof["z_diff_c2_minus_c1"] = Xz[labels == 2].mean() - Xz[labels == 1].mean()
prof = prof.reindex(prof.z_diff_c2_minus_c1.abs().sort_values(ascending=False).index)
print("\nFeature profile (sorted by |z difference| between clusters):")
display(prof.round(3))
print("\nMemberships:")
display(memb.sort_values(["cluster", "silhouette"], ascending=[True, False]).round(3))

# ---- optional: compare with the education superlevel-H0 (TDA) split -----------
# green cluster of pd_clustering_education/clusters_normalized_ot_average_tau0.1.csv
# (sup_h0, k = 2, uniform weights); read from file so it stays in sync with the PD notebook
TDA_CSV = FEATURE_CSV.parent / "pd_clustering_education" / "clusters_normalized_ot_average_tau0.1.csv"
_t = pd.read_csv(TDA_CSV)
_t = _t[(_t.diagram == "sup_h0") & (_t.k == 2)].iloc[0]
TDA_GREEN = {c for c in cities if int(_t[c]) == int(_t["medellin"])}   # cluster containing Medellín
print("TDA green (education sup_h0):", sorted(TDA_GREEN))
tda = np.array([1 if c in TDA_GREEN else 2 for c in cities])
print(f"\nARI (feature clusters vs TDA superlevel-H0 split) = {adjusted_rand_score(tda, labels):.3f}")
display(pd.crosstab(pd.Series(labels, index=cities, name="feature cluster"),
                    pd.Series(np.where(tda == 1, "TDA green", "TDA orange"), index=cities, name="TDA")))

memb.assign(tda_split=np.where(tda == 1, "green", "orange")).to_csv(OUT_DIR / f"memberships_{METHOD}_k{K}.csv")
prof.to_csv(OUT_DIR / f"cluster_profile_{METHOD}_k{K}.csv")

In [ ]:
# ============================================================================
# CELL 2 — How many clusters? (uses X, Xz, F, cities, METHOD, METRIC, OUT_DIR from Cell 1)
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster, cophenet
from scipy.spatial.distance import pdist
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                             davies_bouldin_score, adjusted_rand_score)

K_RANGE = range(2, 9)                     # 17 cities -> k beyond ~8 leaves mostly singletons
METHODS = ["ward", "average", "complete"]
N_BOOT = 500                              # bootstrap resamples for stability
FRAC = 0.8                                # fraction of cities kept in each resample
rng = np.random.default_rng(0)


def cluster(Xm, method, k):
    Zm = linkage(Xm, "ward") if method == "ward" else linkage(pdist(Xm, METRIC), method)
    return fcluster(Zm, k, "maxclust"), Zm


rows, links = [], {}
for method in METHODS:
    _, Zm = cluster(X, method, 2)
    links[method] = Zm
    coph = cophenet(Zm, pdist(X, "euclidean" if method == "ward" else METRIC))[0]
    for k in K_RANGE:
        lab, _ = cluster(X, method, k)
        n_found = len(np.unique(lab))
        sizes = np.bincount(lab)[1:]
        r = dict(method=method, k=k, n_found=n_found, cophenetic=coph,
                 smallest_cluster=int(sizes.min()), n_singletons=int((sizes == 1).sum()))
        if n_found > 1:
            r["silhouette"] = silhouette_score(X, lab, metric="euclidean" if method == "ward" else METRIC)
            r["calinski_harabasz"] = calinski_harabasz_score(X, lab)
            r["davies_bouldin"] = davies_bouldin_score(X, lab)
        # bootstrap stability: re-cluster 80 % subsamples, ARI with the full-data labels on the kept cities
        aris = []
        for _ in range(N_BOOT):
            keep = np.sort(rng.choice(len(X), int(round(FRAC * len(X))), replace=False))
            lab_b, _ = cluster(X[keep], method, k)
            aris.append(adjusted_rand_score(lab[keep], lab_b))
        r["boot_ARI_mean"] = np.mean(aris)
        r["boot_ARI_p10"] = np.quantile(aris, 0.10)
        rows.append(r)

K_TAB = pd.DataFrame(rows)
display(K_TAB.round(3))

# elbow on Ward merge heights: a large jump between consecutive merges suggests a natural cut
Zw = links["ward"]
h = Zw[::-1, 2]                                    # h[0] = last merge (k=1 -> 2), h[1] = (2 -> 3), ...
elbow = pd.DataFrame({"k": np.arange(2, len(h) + 1)[:len(K_RANGE)],
                      "merge_height": h[:len(K_RANGE)],
                      "jump_to_next": (h[:len(K_RANGE)] - h[1:len(K_RANGE) + 1])})

# ---- figure ---------------------------------------------------------------------
fig, axs = plt.subplots(2, 2, figsize=(10, 7))
mk = {"ward": "o-", "average": "s--", "complete": "^:"}
ax = axs[0, 0]
for m in METHODS:
    t = K_TAB[K_TAB.method == m]
    ax.plot(t.k, t.silhouette, mk[m], label=f"{m} (coph. r = {t.cophenetic.iloc[0]:.2f})")
ax.set(xlabel="k", ylabel="Mean silhouette", title="Silhouette (higher = better)")
ax.legend(frameon=False, fontsize=8)

ax = axs[0, 1]
t = K_TAB[K_TAB.method == METHOD]
ax.plot(t.k, t.calinski_harabasz, "o-", color="#2166ac")
ax.set(xlabel="k", title=f"Calinski–Harabasz (blue, higher = better)\nDavies–Bouldin (red, lower = better) — {METHOD}")
ax.set_ylabel("Calinski–Harabasz", color="#2166ac")
ax_b = ax.twinx()
ax_b.plot(t.k, t.davies_bouldin, "s--", color="#b2182b")
ax_b.set_ylabel("Davies–Bouldin", color="#b2182b")

ax = axs[1, 0]
ax.bar(elbow.k, elbow.jump_to_next, color="#777777")
ax.set(xlabel="k", ylabel="Height gap to next merge",
       title="Ward dendrogram gaps (large gap = natural cut at k)")

ax = axs[1, 1]
for m in METHODS:
    t = K_TAB[K_TAB.method == m]
    ax.plot(t.k, t.boot_ARI_mean, mk[m], label=m)
ax.axhline(0.75, ls=":", lw=0.8, color="k")
ax.set(xlabel="k", ylabel="Mean ARI vs full-data labels", ylim=(0, 1.02),
       title=f"Bootstrap stability ({N_BOOT} × {int(FRAC*100)} % of cities)")
ax.legend(frameon=False, fontsize=8)
for a in axs.flat:
    a.set_xticks(list(K_RANGE))
    for s in ("top",):
        a.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "n_clusters_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

# ---- summary and memberships for each k (main method) ----------------------------
best = K_TAB[(K_TAB.method == METHOD) & (K_TAB.n_singletons == 0)]
if len(best):
    b = best.loc[best.silhouette.idxmax()]
    print(f"{METHOD}: best silhouette without singleton clusters at k = {int(b.k)} "
          f"(silhouette = {b.silhouette:.3f}, bootstrap ARI = {b.boot_ARI_mean:.2f})")
MEMB_K = pd.DataFrame({f"k{k}": cluster(X, METHOD, k)[0] for k in K_RANGE}, index=cities)
display(MEMB_K.sort_values(list(MEMB_K.columns)))

K_TAB.to_csv(OUT_DIR / "n_clusters_metrics.csv", index=False)
MEMB_K.to_csv(OUT_DIR / f"memberships_by_k_{METHOD}.csv")

In [ ]:
# CELL 3 — Clustering on the features that separate the TDA split (+ some that do not)
# Uses FEATURE_CSV, TDA_GREEN, METHOD and METRIC from Cell 1.
#
# CAUTION (double dipping): these features are CHOSEN because they differ between the
# TDA groups, so clustering on them will recover the TDA split to some degree by construction.
# The permutation test below asks how much agreement you get from "select the top features,
# then cluster" when the TDA labels are random.  Only agreement above that null is evidence.
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, rankdata
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.metrics import adjusted_rand_score, silhouette_score

Q_MAX = 0.05            # "significant" = BH q below this
N_EXTRA = 5             # add the next N non-significant features (ranked by p) ...
EXTRA_FEATURES = None   # ... or give an explicit list of feature names instead, e.g. ["gini_time", "t_iqr"]
K_SEL = 2
N_PERM = 1000
SEED = 0
OUT_SEL = FEATURE_CSV.parent / "feature_clustering_education" / "selected"
OUT_SEL.mkdir(parents=True, exist_ok=True)

FA = pd.read_csv(FEATURE_CSV, index_col="city")
FA = FA.loc[:, FA.std() > 0]
cities_s = FA.index.tolist()
y = np.array([1 if c in TDA_GREEN else 0 for c in cities_s])
ZA = ((FA - FA.mean()) / FA.std(ddof=0)).to_numpy()
names = FA.columns.to_numpy()

# --- univariate tests on all features
p = np.array([mannwhitneyu(FA[f][y == 1], FA[f][y == 0]).pvalue for f in names])
o_ = np.argsort(p)                              # Benjamini-Hochberg q-values
q = np.empty_like(p); q[o_] = np.minimum.accumulate((p[o_] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1].clip(max=1)
R = rankdata(ZA, axis=0)                       # ranks per feature (for the fast permutation version)

def auc_dev(lab):
    """|AUC - 0.5| per feature; orders features the same way as the Mann-Whitney p-value."""
    n1, n0 = lab.sum(), (1 - lab).sum()
    U = R[lab == 1].sum(0) - n1 * (n1 + 1) / 2
    return np.abs(U / (n1 * n0) - 0.5)

def cluster_ari(cols, lab):
    X = ZA[:, cols]
    Z = linkage(X, method="ward") if METHOD == "ward" else linkage(pdist(X, METRIC), method=METHOD)
    return adjusted_rand_score(lab, fcluster(Z, K_SEL, criterion="maxclust"))

order = np.argsort(-auc_dev(y), kind="stable")      # strongest separators first
sig = [names[i] for i in order if q[i] < Q_MAX]
extra = (list(EXTRA_FEATURES) if EXTRA_FEATURES is not None
         else [names[i] for i in order if q[i] >= Q_MAX][:N_EXTRA])
SETS = {"significant": sig, "significant + extra": sig + [f for f in extra if f not in sig]}
TESTS = pd.DataFrame({"feature": names, "p": p, "q": q,
                      "median_green": FA[y == 1].median().to_numpy(),
                      "median_orange": FA[y == 0].median().to_numpy()}).iloc[order]
TESTS["set"] = np.where(TESTS.feature.isin(sig), "significant",
                        np.where(TESTS.feature.isin(extra), "extra", ""))
TESTS.to_csv(OUT_SEL / "feature_tests_vs_tda.csv", index=False)
print(f"{len(sig)} significant features (q < {Q_MAX}); {len(extra)} extra")

# --- ARI vs number of top-ranked features, real labels and permutation null
rng = np.random.default_rng(SEED)
M = np.arange(2, len(names) + 1)
ari_real = np.array([cluster_ari(order[:m], y) for m in M])
ari_null = np.empty((N_PERM, len(M)))
for b in range(N_PERM):
    yp = rng.permutation(y)
    o = np.argsort(-auc_dev(yp), kind="stable")
    ari_null[b] = [cluster_ari(o[:m], yp) for m in M]
CURVE = pd.DataFrame({"n_features": M, "ari_real": ari_real,
                      "null_median": np.median(ari_null, 0),
                      "null_q95": np.quantile(ari_null, 0.95, 0),
                      "p_perm": ((ari_null >= ari_real).sum(0) + 1) / (N_PERM + 1)})
CURVE.to_csv(OUT_SEL / "ari_vs_n_features.csv", index=False)

# --- the two named sets: real result, silhouette, permutation p (same selection rule under the null)
rows = []
for nm, fs in SETS.items():
    cols = [list(names).index(f) for f in fs]
    X = ZA[:, cols]
    Z = linkage(X, method="ward") if METHOD == "ward" else linkage(pdist(X, METRIC), method=METHOD)
    lab = fcluster(Z, K_SEL, criterion="maxclust")
    m = len(fs)
    rows.append(dict(set=nm, n_features=m, ari=adjusted_rand_score(y, lab),
                     silhouette=silhouette_score(X, lab, metric="euclidean" if METHOD == "ward" else METRIC),
                     p_perm=CURVE.loc[CURVE.n_features == m, "p_perm"].item(),
                     null_q95=CURVE.loc[CURVE.n_features == m, "null_q95"].item(),
                     mismatched=", ".join(c for c, a, t in zip(cities_s, lab, y)
                                          if t != round(y[lab == a].mean()))))   # vs majority TDA group of its cluster
    pd.DataFrame({"cluster": lab, "tda": np.where(y == 1, "green", "orange")},
                 index=cities_s).to_csv(OUT_SEL / f"memberships_{nm.replace(' + ', '_plus_')}_k{K_SEL}.csv")
SUMMARY = pd.DataFrame(rows)
SUMMARY.to_csv(OUT_SEL / "selected_sets_summary.csv", index=False)
display(SUMMARY.round(3))

# --- figure: (a) dendrogram on "significant + extra", (b) z-score heatmap, (c) ARI vs n features
CG, CO = "#1b7837", "#e08214"
fs = SETS["significant + extra"]
cols = [list(names).index(f) for f in fs]
X = ZA[:, cols]
Z = linkage(X, method="ward") if METHOD == "ward" else linkage(pdist(X, METRIC), method=METHOD)
fig = plt.figure(figsize=(12, 8.2))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.15], width_ratios=[1.35, 1], hspace=0.75, wspace=0.28)
a = fig.add_subplot(gs[0, 0]); c = fig.add_subplot(gs[0, 1]); b = fig.add_subplot(gs[1, :])
dn = dendrogram(Z, labels=[s.replace("_", " ").title() for s in cities_s], ax=a,
                link_color_func=lambda k: "#555555", leaf_rotation=90)
a.axhline((Z[-K_SEL, 2] + Z[-(K_SEL - 1), 2]) / 2, color="k", ls="--", lw=0.8)
for t in a.get_xticklabels():
    t.set_color(CG if t.get_text().lower().replace(" ", "_") in TDA_GREEN else CO)
a.set_ylabel(f"{METHOD} linkage height")
a.set_title(f"a  {len(fs)} features ({len(sig)} significant + {len(fs) - len(sig)} extra); "
            f"ARI vs TDA = {SUMMARY.ari.iloc[1]:.2f}\n   leaf colour = TDA group", loc="left", fontsize=9)
a.spines[["top", "right"]].set_visible(False)

leaf = [cities_s[i] for i in dn["leaves"]]
H = pd.DataFrame(X, index=cities_s, columns=fs).loc[leaf].T
im = b.imshow(H.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5)
b.set_yticks(range(len(fs)), [f + ("" if f in sig else "  (extra)") for f in fs], fontsize=7)
b.set_xticks(range(len(leaf)), [s.replace("_", " ").title() for s in leaf], rotation=90, fontsize=7)
for t in b.get_xticklabels():
    t.set_color(CG if t.get_text().lower().replace(" ", "_") in TDA_GREEN else CO)
b.axhline(len(sig) - 0.5, color="k", lw=0.8)
fig.colorbar(im, ax=b, fraction=0.02, pad=0.01, label="z-score")
b.set_title("c  Feature z-scores, cities in dendrogram order", loc="left", fontsize=9)

c.fill_between(M, 0, CURVE.null_q95, color="#bbbbbb", alpha=0.6, lw=0, label="null 95th percentile")
c.plot(M, CURVE.null_median, color="#666666", lw=1, ls="--", label="null median")
c.plot(M, ari_real, color="k", lw=1.4, marker="o", ms=3, label="real TDA labels")
for nm, mk in zip(SETS, ["s", "D"]):
    m = len(SETS[nm]); c.plot(m, ari_real[M == m], mk, color="#c51b7d", ms=6, label=nm)
c.set_xlabel("Number of top-ranked features used"); c.set_ylabel("ARI with TDA split")
c.set_ylim(-0.15, 1.02); c.legend(frameon=False, fontsize=7, loc="upper right")
c.set_title(f"b  Select-then-cluster vs {N_PERM} label permutations", loc="left", fontsize=9)
c.spines[["top", "right"]].set_visible(False)
fig.savefig(OUT_SEL / f"selected_features_{METHOD}_k{K_SEL}.png", dpi=300, bbox_inches="tight")
plt.show()
